In [1]:
import pandas as pd

In [31]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
stk_data=pd.read_csv("Tatacoffee13_21.csv",parse_dates=['Date'],index_col='Date')

In [7]:
stk_data

,Open,High,Low,Close
Date,,,,
2013-01-01,1410.60,1427.90,1408.30,1415.10
2013-01-02,1421.00,1626.60,1416.15,1607.40
2013-01-03,1632.55,1673.90,1613.05,1626.20
2013-01-04,1627.75,1627.75,1574.60,1579.05
2013-01-07,1580.00,1639.50,1565.50,1595.65
...,...,...,...,...
2021-12-22,202.90,207.80,201.35,205.00
2021-12-23,206.00,206.85,202.05,202.95
2021-12-24,203.90,203.90,199.35,201.00


In [9]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (2225, 4)


In [10]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [11]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

1780
X_train length: (1780, 4)
X_test length: (445, 4)
y_train length: (1780, 4)
y_test length: (445, 4)


In [15]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Order":[],"Test":[]} 

In [17]:
def cominbation(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
    test_obs = 28
    train =datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    from statsmodels.tsa.api import VARMAX
    orders=[(1,2),(1,1)]
    best_aic = float("inf")
    best_order = None
    for i in orders:
        model = VARMAX(train,order=i)
        results = model.fit()
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
        if results.aic < best_aic:
            best_aic = results.aic
            best_order = i
    model = VARMAX(train,order=best_order)
    results = model.fit()
    pred=results.forecast(steps=test_obs)
    preds=pd.DataFrame(pred,columns=listt)
    from sklearn.metrics import root_mean_squared_error
    rmse= root_mean_squared_error(test,pred)
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test,pred)
    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Order"].append(best_order)
    performance["Test"].append(test_obs)
    perf=pd.DataFrame(performance)
    return perf,results,pred
        

In [41]:
listt=["Close","High","Low","Open"]

In [43]:
perf,results,pred=cominbation(data1,listt)

['Close', 'High', 'Low', 'Open']
Order = (1, 2)
AIC:  -66347.37276967264
BIC:  -65994.2921891848

Order = (1, 1)
AIC:  -66334.65928035788
BIC:  -66072.6962690282



In [45]:
perf

,Model,RMSE,MaPe,Order,Test
0,"[Close, High]",0.016416,0.158240,"(1, 2)",28
1,"[Close, High, Low]",0.016694,0.161349,"(1, 1)",28
2,"[Close, High, Low, Open]",0.016405,0.157661,"(1, 2)",28
